In [1]:
import numpy as np
from scipy.sparse import random as sparse_random, csr_matrix, diags
import matplotlib.pyplot as plt
import numba as nb
from scipy.sparse.linalg import eigsh
import time

def generate_sparse_matrix(size, density):
    diagonal = np.linspace(-1, 1, size)
    rvs = np.random.randn  # SciPy will call with a size -> returns N(0,1) samples
    upper = sparse_random(size, size, density=density/2, data_rvs=rvs, format='coo')
    A = upper + upper.T
    A.setdiag(diagonal)
    return A.tocsr()

@nb.njit(parallel=True, fastmath=True, cache=True)
def block_csr_mv(A_data, A_indices, A_indptr, X):
    n = A_indptr.shape[0] - 1
    b = X.shape[1]
    Y = np.zeros((n, b), dtype=X.dtype)
    for i in nb.prange(n):
        for j in range(A_indptr[i], A_indptr[i+1]):
            col = A_indices[j]
            for k in range(b):
                Y[i, k] += A_data[j] * X[col, k]
    return Y

def _gamma_from_sorted(vals, clip=1e9):
    if vals.shape[0] < 3:
        return None
    lam1, lam2, lam_max = vals[0], vals[1], vals[-1]
    denom = lam2 - lam1
    if denom <= 0:
        return None
    g = (lam_max - lam2) / denom
    if not np.isfinite(g):
        return None
    return min(g, clip)

def _true_gamma_eigsh(A, clip=1e9):
    try:
        lam_small = np.sort(eigsh(A, k=2, which='SA', return_eigenvectors=False))
        lam1, lam2 = lam_small[0], lam_small[1]
        lam_max = eigsh(A, k=1, which='LA', return_eigenvectors=False)[0]
        denom = lam2 - lam1
        if denom <= 0:
            return None
        g = (lam_max - lam2) / denom
        if not np.isfinite(g):
            return None
        return min(g, clip)
    except Exception:
        return None

@nb.njit(fastmath=True, cache=True)
def block_lanczos_loop(A_data, A_indices, A_indptr, Q, T_blocks, m, b, ortho_thresh):
    n = Q.shape[0]
    eps = np.finfo(np.float64).eps
    m_eff = m  
    for k in range(m):
        start = k * b
        end = (k+1) * b
        Qk = Q[:, start:end]

        V = block_csr_mv(A_data, A_indices, A_indptr, Qk)

        Qk_contig = np.ascontiguousarray(Qk)
        V_contig = np.ascontiguousarray(V)
        alpha = np.dot(Qk_contig.T, V_contig)

        for i in range(b):
            for j in range(b):
                T_blocks[k, k, i, j] = alpha[i, j]

        V = V - np.dot(Qk_contig, np.ascontiguousarray(alpha))

        if k > 0:
            Q_prev = Q[:, (k-1)*b : k*b]
            beta_prev = T_blocks[k, k-1]  # (b,b)
            V = V - np.dot(np.ascontiguousarray(Q_prev), np.ascontiguousarray(beta_prev))

        norm_V = np.sqrt(np.sum(V * V))
        dyn_thresh = max(ortho_thresh, np.sqrt(eps) * norm_V)
        for rep in range(2):
            for j in range(k+1):
                Qj = Q[:, j*b:(j+1)*b]
                P = np.dot(np.ascontiguousarray(Qj).T, np.ascontiguousarray(V))
                proj_norm = np.sqrt(np.sum(P * P))  # Frobenius
                if proj_norm > dyn_thresh:
                    V = V - np.dot(np.ascontiguousarray(Qj), P)

        Q_new, R = np.linalg.qr(V)
        next_start = (k+1)*b
        next_end = (k+2)*b
        Q[:, next_start:next_end] = Q_new

        T_blocks[k+1, k] = R 

        rnorm = np.sqrt(np.sum(R * R))
        if rnorm == 0:
            m_eff = k + 1
            return m_eff

    return m_eff

# ===================== IRL (your implementation) =====================
def block_irl(matrix_s, v_init, tol=1e-8, m=None, max_iter=40000, max_m=100,ortho_thresh=1e-10, k=1, record_gamma=True, compute_true_gamma=False,
              gamma_clip=1e6):
    n = matrix_s.shape[0]
    if m is None:
        m = max(2*k+1, 40)

    Q = np.zeros((n, (m+1)*k), dtype=np.float64)
    T_blocks = np.zeros((m+1, m, k, k), dtype=np.float64)

    if v_init.ndim == 1:
        v_init = v_init.reshape(-1, 1)
        if k > 1:
            v_init = np.repeat(v_init, k, axis=1)
    Q0, _ = np.linalg.qr(v_init)
    Q[:, :k] = Q0[:, :k]

    A = matrix_s.tocsr()
    A_data, A_indices, A_indptr = A.data, A.indices, A.indptr

    total_iters = 0
    instantaneous_sizes = []
    last_ritz_res = np.inf

    gamma_true = None
    gamma_estimates = []
    if compute_true_gamma:
        gamma_true = _true_gamma_eigsh(A, clip=gamma_clip)

    while total_iters < max_iter:
        m_eff = block_lanczos_loop(A_data, A_indices, A_indptr, Q, T_blocks, m, k, ortho_thresh)
        total_iters += m_eff
        instantaneous_sizes.append(m_eff * k)

        T_dense = np.zeros((m_eff*k, m_eff*k), dtype=np.float64)
        for i in range(m_eff):
            T_dense[i*k:(i+1)*k, i*k:(i+1)*k] = T_blocks[i, i]
            if i < m_eff - 1:
                B = T_blocks[i+1, i]
                T_dense[(i+1)*k:(i+2)*k, i*k:(i+1)*k] = B
                T_dense[i*k:(i+1)*k, (i+1)*k:(i+2)*k] = B.T

        eigvals, eigvecs = np.linalg.eigh(T_dense)  # ascending

        if record_gamma:
            g_est = _gamma_from_sorted(eigvals, clip=gamma_clip)
            gamma_estimates.append(g_est)

        if m_eff < 1:
            ritz_res = np.inf
        else:
            Bm = T_blocks[m_eff, m_eff-1]  # (k,k)
            if k == 1:
                y_last = eigvecs[-1, 0]
                beta = Bm[0, 0]
                ritz_res = abs(beta * y_last)
            else:
                ritz_res = 0.0
                for j in range(k):
                    y_tail = eigvecs[-k:, j]
                    rj = np.linalg.norm(Bm @ y_tail)
                    if rj > ritz_res:
                        ritz_res = rj

        lam_target = eigvals[0]
        if ritz_res < tol * max(1.0, abs(lam_target)):
            approx_eigvecs = Q[:, :m_eff*k] @ eigvecs[:, :k]
            return total_iters, eigvals[:k], approx_eigvecs, instantaneous_sizes, gamma_true, gamma_estimates

        target_ind = 0
        unwanted = np.delete(eigvals, np.arange(target_ind, target_ind+k))
        shifts = np.sort(unwanted)[::-1] if unwanted.size else np.array([])

        V_plus = np.eye(m_eff*k)
        T_temp = T_dense.copy()
        for mu in shifts:
            Qs, Rs = np.linalg.qr(T_temp - mu * np.eye(m_eff*k))
            T_temp = Rs @ Qs + mu * np.eye(m_eff*k)
            V_plus = V_plus @ Qs

        new_block = Q[:, :m_eff*k] @ V_plus[:, :k]
        Q_new, _ = np.linalg.qr(new_block)
        Q[:, :k] = Q_new

        if ritz_res > 0.8 * last_ritz_res and m < max_m:
            new_m = min(int(m * 1.15), max_m)
            if new_m > m:
                m = new_m
                new_Q = np.zeros((n, (m+1)*k), dtype=np.float64)
                new_Q[:, :k] = Q[:, :k]
                Q = new_Q
                T_blocks = np.zeros((m+1, m, k, k), dtype=np.float64)

        last_ritz_res = ritz_res

    approx_eigvecs = Q[:, :m_eff*k] @ eigvecs[:, :k]
    return total_iters, eigvals[:k], approx_eigvecs, instantaneous_sizes, gamma_true, gamma_estimates

# ===================== Warmup & fast gamma =====================
def _warmup_numba():
    n, b = 16, 1
    from scipy.sparse import diags as _diags
    A = _diags([np.ones(n-1), 2*np.ones(n), np.ones(n-1)], [-1,0,1]).tocsr()
    A_data, A_indices, A_indptr = A.data, A.indices, A.indptr
    X = np.random.randn(n, b)
    _ = block_csr_mv(A_data, A_indices, A_indptr, X)
    m = 4
    Q = np.zeros((n, (m+1)*b))
    Q[:, :b] = np.linalg.qr(np.random.randn(n, b))[0]
    T_blocks = np.zeros((m+1, m, b, b))
    _ = block_lanczos_loop(A_data, A_indices, A_indptr, Q, T_blocks, m, b, 1e-12)

def _true_gamma_eigsh_fast(A, clip=1e9, tol=1e-8, maxiter=4000):
    try:
        lam_small = np.sort(eigsh(A, k=2, which='SA', return_eigenvectors=False, tol=tol, maxiter=maxiter))
        lam1, lam2 = lam_small[0], lam_small[1]
        lam_max = eigsh(A, k=1, which='LA', return_eigenvectors=False, tol=tol, maxiter=maxiter)[0]
        denom = lam2 - lam1
        if denom <= 0:
            return None
        g = (lam_max - lam2) / denom
        if not np.isfinite(g):
            return None
        return min(g, clip)
    except Exception:
        return None

# ===================== NEW: constant-gamma apparatus =====================
def make_sparse_symmetric_S(n, density, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    nnz_density = max(0.0, min(1.0, density/2.0))
    rvs = rng.standard_normal
    U = sparse_random(n, n, density=nnz_density, data_rvs=rvs, format='coo', random_state=rng)
    S = U + U.T
    S.setdiag(0.0)
    return S.tocsr()

def gamma_true_fast(A, clip=1e9, tol=1e-8, maxiter=4000):
    return _true_gamma_eigsh_fast(A, clip=clip, tol=tol, maxiter=maxiter)

def tune_scale_for_target_gamma(S, diag_vals, gamma_target, tol_frac=0.1,s_min=0.05, s_max=5.0, ngrid=12, max_refines=2,
                                gamma_tol=1e-8, gamma_maxiter=4000):
    def A_of(s):
        return (diags(diag_vals).tocsr() + (S * s)).tocsr()

    best = (None, None, np.inf)  # (s, gamma, rel_err)
    s_lo, s_hi = s_min, s_max
    for _ in range(max_refines + 1):
        grid = np.logspace(np.log10(s_lo), np.log10(s_hi), ngrid)
        for s in grid:
            g = gamma_true_fast(A_of(s), tol=gamma_tol, maxiter=gamma_maxiter)
            if g is None:
                continue
            rel = abs(g - gamma_target) / max(gamma_target, 1e-12)
            if rel < best[2]:
                best = (s, g, rel)
        if best[0] is None:
            break
        s_star = best[0]
        s_lo, s_hi = max(s_min, s_star/3.0), min(s_max, s_star*3.0)
        if best[2] <= tol_frac:
            break
    return best  # s_best, gamma_best, rel_err

def evaluate_constant_gamma_across_densities(size, density_levels,gamma_target,num_trials=10, k=1,irl_tol=1e-8, irl_max_iter=10000,ortho_thresh=1e-10,
                                             gamma_tol=1e-8, gamma_maxiter=4000,
                                             tune_tol_frac=0.10,
                                             rng_seed=0):
    _warmup_numba()
    rng = np.random.default_rng(rng_seed)
    diag_vals = np.linspace(-1.0, 1.0, size)

    rows = []
    for dens in density_levels:
        print(f"\n--- Evaluating Density Level: {dens*100:.2f}% ---", flush=True)
        g_true_list, g_irl_list, basis_list, rel_err_list = [], [], [], []
        for _ in range(num_trials):
            S = make_sparse_symmetric_S(size, dens, rng=rng)
            s_best, g_best, rel_err = tune_scale_for_target_gamma(
                S, diag_vals, gamma_target,
                tol_frac=tune_tol_frac, s_min=0.05, s_max=5.0, ngrid=12, max_refines=2,
                gamma_tol=gamma_tol, gamma_maxiter=gamma_maxiter
            )
            if s_best is None:
                continue
            A = (diags(diag_vals).tocsr() + (S * s_best)).tocsr()
            g_true_list.append(g_best); rel_err_list.append(rel_err)

            v0 = rng.random((size, k))
            total_iters, _, _, inst_sizes, _, gamma_estimates = block_irl(
                A, v0, tol=irl_tol, m=None, max_iter=irl_max_iter,
                max_m=100, ortho_thresh=ortho_thresh, k=k,
                record_gamma=True, compute_true_gamma=False
            )
            basis_list.append(int(np.sum(inst_sizes)))
            # last valid IRL gamma estimate
            g_irl = np.nan
            for ge in reversed(gamma_estimates):
                if ge is not None and np.isfinite(ge):
                    g_irl = float(ge); break
            g_irl_list.append(g_irl)

        avg_g_true = float(np.nanmean(g_true_list)) if len(g_true_list) else np.nan
        std_g_true = float(np.nanstd(g_true_list))  if len(g_true_list) else np.nan
        avg_g_irl  = float(np.nanmean(g_irl_list))  if len(g_irl_list)  else np.nan
        std_g_irl  = float(np.nanstd(g_irl_list))   if len(g_irl_list)  else np.nan
        avg_basis  = float(np.nanmean(basis_list))  if len(basis_list)  else np.nan
        std_basis  = float(np.nanstd(basis_list))   if len(basis_list)  else np.nan
        avg_relerr = float(np.nanmean(rel_err_list)) if len(rel_err_list) else np.nan

        print(f"Density {dens*100:6.2f}%  constant γ target={gamma_target:.5g}  "
              f"γ_true={avg_g_true:.4g}  "
              f"γ_IRL={avg_g_irl:.4g}  "
              f"Lanczos basis size={avg_basis:.1f}  ")

        rows.append({
            'Density': dens,
            'gamma_target': gamma_target,
            'gamma_true_avg': avg_g_true, 'gamma_true_std': std_g_true,
            'gamma_irl_avg':  avg_g_irl,  'gamma_irl_std':  std_g_irl,
            'basis_avg':      avg_basis,  'basis_std':      std_basis,
        })
    return rows

# ===================== (Optional) Original comparison driver =====================
def evaluate_gamma_true_and_irl(size, density_levels, num_trials=1, k=1,
                                tol_irl=1e-8, max_iter_irl=10000,
                                ortho_thresh=1e-10,
                                gamma_tol=1e-8, gamma_maxiter=10000,
                                print_live=True):
    _warmup_numba()
    avg_table = {}

    for density in density_levels:
        if print_live:
            print(f"\n--- Evaluating Density Level: {density*100:.2f}% ---", flush=True)
        g_true_list, g_irl_list, basis_list = [], [], []

        for _ in range(num_trials):
            A  = generate_sparse_matrix(size, density)
            v0 = np.random.rand(size, k)

            g_true = _true_gamma_eigsh_fast(A, clip=1e9, tol=gamma_tol, maxiter=gamma_maxiter)

            total_iters, _, _, inst_sizes, _, gamma_estimates = block_irl(
                A, v0, tol=tol_irl, m=None, max_iter=max_iter_irl, max_m=100,
                ortho_thresh=ortho_thresh, k=k,
                record_gamma=True, compute_true_gamma=False
            )
            basis = float(np.sum(inst_sizes))

            g_irl = None
            for ge in reversed(gamma_estimates):
                if ge is not None and np.isfinite(ge):
                    g_irl = float(ge); break

            if g_true is not None and np.isfinite(g_true):
                g_true_list.append(float(g_true))
            if g_irl is not None:
                g_irl_list.append(float(g_irl))
            basis_list.append(basis)

        avg_gamma_true = float(np.mean(g_true_list)) if len(g_true_list) else float('nan')
        avg_gamma_irl  = float(np.mean(g_irl_list))  if len(g_irl_list)  else float('nan')
        avg_basis      = float(np.mean(basis_list))  if len(basis_list)  else float('nan')

        avg_table[density] = {
            'avg_gamma_true': avg_gamma_true,
            'avg_gamma_irl':  avg_gamma_irl,
            'avg_basis':      avg_basis,
            'n_used_true':    len(g_true_list),
            'n_used_irl':     len(g_irl_list),
            'n_trials':       num_trials
        }

        if print_live:
            dens_pct = 100.0 * density
            gT = f"{avg_gamma_true:.6g}" if np.isfinite(avg_gamma_true) else "nan"
            gI = f"{avg_gamma_irl:.6g}"  if np.isfinite(avg_gamma_irl)  else "nan"
            bA = f"{avg_basis:.6g}"      if np.isfinite(avg_basis)      else "nan"
            print(f"Density {dens_pct:6.2f}%:  avg γ(true) = {gT} , avg γ(IRL) = {gI}  ,  avg basis = {bA}")

    return avg_table


size = 10000
density_levels = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45,0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
k = 1

# A) Constant-γ sweep (recommended for your professor's question)
gamma_target = 1000          # ← choose a target γ to hold roughly constant
rows = evaluate_constant_gamma_across_densities(
    size=size,
    density_levels=density_levels,
    gamma_target=gamma_target,
    num_trials=10,            # average a few random shapes S per density
    k=k,
    irl_tol=1e-8, irl_max_iter=10000,
    ortho_thresh=1e-10,
    gamma_tol=1e-8, gamma_maxiter=4000,   # fast γ during tuning
    tune_tol_frac=0.10,
    rng_seed=42
    )

    # B) (Optional) The original “no tuning” comparison
    # avg_table_both = evaluate_gamma_true_and_irl(
    #     size, density_levels,
    #     num_trials=20, k=k,
    #     tol_irl=1e-8, max_iter_irl=10000,
    #     ortho_thresh=1e-10,
    #     gamma_tol=1e-3, gamma_maxiter=300,
    #     print_live=True
    # )



--- Evaluating Density Level: 1.00% ---
Density   1.00%  constant γ target=1000  γ_true=953.6  γ_IRL=741.2  Lanczos basis size=334.6  

--- Evaluating Density Level: 5.00% ---
Density   5.00%  constant γ target=1000  γ_true=1040  γ_IRL=804.3  Lanczos basis size=331.6  

--- Evaluating Density Level: 10.00% ---
Density  10.00%  constant γ target=1000  γ_true=962.9  γ_IRL=698.1  Lanczos basis size=335.0  

--- Evaluating Density Level: 15.00% ---
Density  15.00%  constant γ target=1000  γ_true=1018  γ_IRL=721.4  Lanczos basis size=352.4  

--- Evaluating Density Level: 20.00% ---
Density  20.00%  constant γ target=1000  γ_true=903.4  γ_IRL=689.1  Lanczos basis size=335.6  

--- Evaluating Density Level: 25.00% ---
Density  25.00%  constant γ target=1000  γ_true=1102  γ_IRL=805.2  Lanczos basis size=363.4  

--- Evaluating Density Level: 30.00% ---
Density  30.00%  constant γ target=1000  γ_true=915.1  γ_IRL=583.5  Lanczos basis size=316.0  

--- Evaluating Density Level: 35.00% ---
Dens